# User Satisfaction Analysis 
Assuming that the satisfaction of a user is dependent on user engagement and experience, you’re expected in this section to analyze customer satisfaction in depth. The following tasks will guide you: 

Based on the engagement analysis + the experience analysis you conducted above,

In [1]:
# Import Dataset and useful libraries 
import os 
import pandas as pd 
os.chdir('../scripts/')
import utils as util 


data_path = "../../data/week2/source.csv"
df = util.read_csv_file(data_path)
data = df.get("data")

In [5]:
from sklearn.cluster import KMeans

# Selecting relevant features for clustering
features = data[['Avg RTT DL (ms)', 'Avg Bearer TP DL (kbps)', 'TCP DL Retrans. Vol (Bytes)']].copy()

# Handling NaN values by filling with the mean
features.fillna(features.mean(), inplace=True)

# Performing k-means clustering for engagement and experience
kmeans_engagement = KMeans(n_clusters=3, random_state=42)
data.loc[:, 'engagement_cluster'] = kmeans_engagement.fit_predict(features)

kmeans_experience = KMeans(n_clusters=3, random_state=42)
data.loc[:, 'experience_cluster'] = kmeans_experience.fit_predict(features)

# Calculate engagement score as Euclidean distance to the less engaged cluster centroid
engagement_cluster_centroid = kmeans_engagement.cluster_centers_[data['engagement_cluster']]
data.loc[:, 'engagement_score'] = ((data[['Avg RTT DL (ms)', 'Avg Bearer TP DL (kbps)', 'TCP DL Retrans. Vol (Bytes)']] - engagement_cluster_centroid) ** 2).sum(axis=1) ** 0.5

# Calculate experience score as Euclidean distance to the worst experience cluster centroid
experience_cluster_centroid = kmeans_experience.cluster_centers_[data['experience_cluster']]
data.loc[:, 'experience_score'] = ((data[['Avg RTT DL (ms)', 'Avg Bearer TP DL (kbps)', 'TCP DL Retrans. Vol (Bytes)']] - experience_cluster_centroid) ** 2).sum(axis=1) ** 0.5

data.head()

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
c:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


,Bearer Id,Start,Start ms,End,End ms,Dur. (ms),IMSI,MSISDN/Number,IMEI,Last Location Name,...,Gaming DL (Bytes),Gaming UL (Bytes),Other DL (Bytes),Other UL (Bytes),Total UL (Bytes),Total DL (Bytes),engagement_cluster,experience_cluster,engagement_score,experience_score
0,1.311448e+19,4/4/2019 12:01,770.0,4/25/2019 14:35,662.0,1823652.0,2.082014e+14,3.366496e+10,3.552121e+13,9.16456699548519E+015,...,278082303.0,14344150.0,171744450.0,8814393.0,36749741.0,308879636.0,0,0,13156.037403,13156.037403
1,1.311448e+19,4/9/2019 13:04,235.0,4/25/2019 8:15,606.0,1365104.0,2.082019e+14,3.368185e+10,3.579401e+13,L77566A,...,608750074.0,1170709.0,526904238.0,15055145.0,53800391.0,653384965.0,0,0,13162.938967,13162.938967
2,1.311448e+19,4/9/2019 17:42,1.0,4/25/2019 11:58,652.0,1361762.0,2.082003e+14,3.376063e+10,3.528151e+13,D42335A,...,229584621.0,395630.0,410692588.0,4215763.0,27883638.0,279807335.0,0,0,13172.862788,13172.862788
3,1.311448e+19,4/10/2019 0:31,486.0,4/25/2019 7:36,171.0,1321509.0,2.082014e+14,3.375034e+10,3.535661e+13,T21824A,...,799538153.0,10849722.0,749039933.0,12797283.0,43324218.0,846028530.0,0,0,13134.862788,13134.862788
4,1.311448e+19,4/12/2019 20:10,565.0,4/25/2019 10:40,954.0,1089009.0,2.082014e+14,3.369980e+10,3.540701e+13,D88865A,...,527707248.0,3529801.0,550709500.0,13910322.0,38542814.0,569138589.0,0,0,13172.862788,13172.862788


In [6]:
# Calculate satisfaction score as the average of engagement and experience scores
data['satisfaction_score'] = (data['engagement_score'] + data['experience_score']) / 2

# Get the top 10 satisfied customers based on the satisfaction score
top_satisfied_customers = data.nlargest(10, 'satisfaction_score')

# Display the top 10 satisfied customers
top_satisfied_customers[['Bearer Id', 'satisfaction_score']]


,Bearer Id,satisfaction_score
77950,7.277826e+18,1.072676e+09
135677,1.304243e+19,1.069631e+09
34636,1.304243e+19,1.068127e+09
140797,7.277826e+18,1.067738e+09
3741,7.277826e+18,1.066311e+09
119667,1.304243e+19,1.053510e+09
39608,7.277826e+18,1.046682e+09
76971,1.304243e+19,1.038247e+09
59011,7.277826e+18,1.034900e+09
41182,1.304243e+19,1.032894e+09


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Define features and target variable
X = data[['engagement_score', 'experience_score']]
y = data['satisfaction_score']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a linear regression model
model = LinearRegression()

# Fit the model to the training data
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Print the evaluation metrics
print(f'Mean Squared Error: {mse}')
print(f'R^2 Score: {r2}')


Mean Squared Error: 5.336764007869734e-15
R^2 Score: 1.0


In [8]:
from sklearn.cluster import KMeans

# Prepare the data for clustering
X_cluster = data[['engagement_score', 'experience_score']]

# Create a KMeans model with 2 clusters
kmeans = KMeans(n_clusters=2, random_state=42)

# Fit the model to the data
data['engagement_cluster'] = kmeans.fit_predict(X_cluster)

# Display the first few rows with the cluster assignments
print(data[['engagement_score', 'experience_score', 'engagement_cluster']].head())


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


   engagement_score  experience_score  engagement_cluster
0      13156.037403      13156.037403                   0
1      13162.938967      13162.938967                   0
2      13172.862788      13172.862788                   0
3      13134.862788      13134.862788                   0
4      13172.862788      13172.862788                   0


In [9]:
# Aggregate the average satisfaction and experience score per cluster
cluster_averages = data.groupby('engagement_cluster').agg(
    average_satisfaction_score=('satisfaction_score', 'mean'),
    average_experience_score=('experience_score', 'mean')
).reset_index()

# Display the aggregated results
print(cluster_averages)


   engagement_cluster  average_satisfaction_score  average_experience_score
0                   0                6.580311e+06              6.580311e+06
1                   1                4.639355e+08              4.639355e+08
